<a href="https://colab.research.google.com/github/Aymane-Aziz/toxic-comment-classifier/blob/main/04_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install wandb for experiment tracking
!pip install wandb -q

from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import f1_score, roc_auc_score
import wandb

print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
GPU available: True
Device: Tesla T4


In [2]:
# All hyperparameters live in one place — easy to change and track
CONFIG = {
    'model_name'   : 'roberta-base',
    'max_len'      : 128,
    'batch_size'   : 32,
    'epochs'       : 3,
    'learning_rate': 2e-5,       # standard starting LR for transformers
    'warmup_steps' : 100,        # gradual LR warmup to stabilize early training
    'base_path'    : '/content/drive/MyDrive/toxic-comment-classifier/',
}

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Config ready, device:", DEVICE)

Config ready, device: cuda


In [3]:
class ToxicCommentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data      = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text   = str(self.data['cleaned_text'][idx])
        labels = self.data[LABEL_COLS].iloc[idx].values.astype(float)

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids'     : encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels'        : torch.tensor(labels, dtype=torch.float)
        }

In [4]:
BASE_PATH = CONFIG['base_path'] + 'data/'

train_df = pd.read_csv(BASE_PATH + 'train_processed.csv')
val_df   = pd.read_csv(BASE_PATH + 'val_processed.csv')

# Fill NaNs just in case
train_df['cleaned_text'] = train_df['cleaned_text'].fillna('')
val_df['cleaned_text']   = val_df['cleaned_text'].fillna('')

tokenizer = RobertaTokenizer.from_pretrained(CONFIG['model_name'])

train_dataset = ToxicCommentDataset(train_df, tokenizer, CONFIG['max_len'])
val_dataset   = ToxicCommentDataset(val_df,   tokenizer, CONFIG['max_len'])

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train batches: 4239
Val batches:   748


In [5]:
model = RobertaForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=6,           # one output per label
    problem_type='multi_label_classification'
)

model = model.to(DEVICE)
print("Model loaded and moved to", DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to cuda
Parameters: 124,650,246


In [6]:
# Load saved class weights from Phase 3
pos_weights = torch.load(BASE_PATH + 'pos_weights.pt').to(DEVICE)

# BCEWithLogitsLoss = Binary Cross Entropy — perfect for multi-label problems
# pos_weight tells the loss to penalize missing rare labels more heavily
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# AdamW is the standard optimizer for transformers
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=0.01)

# Scheduler gradually reduces learning rate during training
total_steps = len(train_loader) * CONFIG['epochs']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=CONFIG['warmup_steps'],
    num_training_steps=total_steps
)

print("Loss, optimizer and scheduler ready")

Loss, optimizer and scheduler ready


In [7]:
wandb.login()  # it will ask for your API key — get it from wandb.ai/settings

run = wandb.init(
    project='toxic-comment-classifier',
    config=CONFIG,
    name='roberta-base-run-1'
)

print("wandb initialized!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aymane-aziz2004 (aymane-aziz2004-aymane) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb initialized!


In [8]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()               # clear previous gradients
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)
        loss.backward()                     # compute gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevent exploding gradients
        optimizer.step()                    # update weights
        scheduler.step()                    # update learning rate

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(loader)} — Loss: {loss.item():.4f}")

    return total_loss / len(loader)

In [9]:
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():   # no gradient computation needed during evaluation
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)
            total_loss += loss.item()

            preds = torch.sigmoid(outputs.logits).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    # Convert probabilities to binary predictions at 0.5 threshold
    binary_preds = (all_preds > 0.5).astype(int)

    f1      = f1_score(all_labels, binary_preds, average='macro', zero_division=0)
    roc_auc = roc_auc_score(all_labels, all_preds, average='macro')

    return total_loss / len(loader), f1, roc_auc

In [10]:
best_roc_auc   = 0
SAVE_PATH      = CONFIG['base_path'] + 'checkpoints/'
os.makedirs(SAVE_PATH, exist_ok=True)

for epoch in range(CONFIG['epochs']):
    print(f"\n{'='*50}")
    print(f"EPOCH {epoch+1}/{CONFIG['epochs']}")
    print(f"{'='*50}")

    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
    val_loss, val_f1, val_roc_auc = eval_epoch(model, val_loader, criterion, DEVICE)

    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val F1:     {val_f1:.4f}")
    print(f"Val ROC-AUC:{val_roc_auc:.4f}")

    # Log to wandb
    wandb.log({
        'epoch'      : epoch + 1,
        'train_loss' : train_loss,
        'val_loss'   : val_loss,
        'val_f1'     : val_f1,
        'val_roc_auc': val_roc_auc
    })

    # Save best model
    if val_roc_auc > best_roc_auc:
        best_roc_auc = val_roc_auc
        model.save_pretrained(SAVE_PATH + 'best_model')
        tokenizer.save_pretrained(SAVE_PATH + 'best_model')
        print(f"✅ New best model saved! ROC-AUC: {best_roc_auc:.4f}")

wandb.finish()
print(f"\nTraining complete! Best ROC-AUC: {best_roc_auc:.4f}")


EPOCH 1/3
  Batch 0/4239 — Loss: 1.1001
  Batch 100/4239 — Loss: 0.1183
  Batch 200/4239 — Loss: 0.2271
  Batch 300/4239 — Loss: 0.1967
  Batch 400/4239 — Loss: 0.1245
  Batch 500/4239 — Loss: 0.0651
  Batch 600/4239 — Loss: 0.5569
  Batch 700/4239 — Loss: 0.1762
  Batch 800/4239 — Loss: 0.1626
  Batch 900/4239 — Loss: 0.2072
  Batch 1000/4239 — Loss: 0.1587
  Batch 1100/4239 — Loss: 0.2760
  Batch 1200/4239 — Loss: 0.0998
  Batch 1300/4239 — Loss: 0.0577
  Batch 1400/4239 — Loss: 0.4438
  Batch 1500/4239 — Loss: 0.1290
  Batch 1600/4239 — Loss: 0.1769
  Batch 1700/4239 — Loss: 0.1397
  Batch 1800/4239 — Loss: 0.1082
  Batch 1900/4239 — Loss: 0.1550
  Batch 2000/4239 — Loss: 0.1977
  Batch 2100/4239 — Loss: 0.0397
  Batch 2200/4239 — Loss: 0.4283
  Batch 2300/4239 — Loss: 0.1352
  Batch 2400/4239 — Loss: 0.1050
  Batch 2500/4239 — Loss: 0.0671
  Batch 2600/4239 — Loss: 0.0941
  Batch 2700/4239 — Loss: 0.0446
  Batch 2800/4239 — Loss: 0.0365
  Batch 2900/4239 — Loss: 0.2191
  Batch 300

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ New best model saved! ROC-AUC: 0.9867

EPOCH 2/3
  Batch 0/4239 — Loss: 0.1666
  Batch 100/4239 — Loss: 0.7497
  Batch 200/4239 — Loss: 0.0209
  Batch 300/4239 — Loss: 0.0834
  Batch 400/4239 — Loss: 0.1225
  Batch 500/4239 — Loss: 0.0277
  Batch 600/4239 — Loss: 0.0668
  Batch 700/4239 — Loss: 0.0040
  Batch 800/4239 — Loss: 0.0102
  Batch 900/4239 — Loss: 0.1711
  Batch 1000/4239 — Loss: 1.3496
  Batch 1100/4239 — Loss: 0.0643
  Batch 1200/4239 — Loss: 0.0451
  Batch 1300/4239 — Loss: 0.0369
  Batch 1400/4239 — Loss: 0.0443
  Batch 1500/4239 — Loss: 0.0276
  Batch 1600/4239 — Loss: 0.0835
  Batch 1700/4239 — Loss: 0.0779
  Batch 1800/4239 — Loss: 0.5138
  Batch 1900/4239 — Loss: 0.0274
  Batch 2000/4239 — Loss: 0.0587
  Batch 2100/4239 — Loss: 0.0072
  Batch 2200/4239 — Loss: 0.0747
  Batch 2300/4239 — Loss: 0.0227
  Batch 2400/4239 — Loss: 0.2278
  Batch 2500/4239 — Loss: 0.1123
  Batch 2600/4239 — Loss: 0.1151
  Batch 2700/4239 — Loss: 0.0303
  Batch 2800/4239 — Loss: 0.5955
  Ba

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ New best model saved! ROC-AUC: 0.9880

EPOCH 3/3
  Batch 0/4239 — Loss: 0.1227
  Batch 100/4239 — Loss: 0.0497
  Batch 200/4239 — Loss: 0.0306
  Batch 300/4239 — Loss: 0.0983
  Batch 400/4239 — Loss: 0.0611
  Batch 500/4239 — Loss: 0.0931
  Batch 600/4239 — Loss: 0.0334
  Batch 700/4239 — Loss: 0.0407
  Batch 800/4239 — Loss: 0.2161
  Batch 900/4239 — Loss: 0.0173
  Batch 1000/4239 — Loss: 0.1292
  Batch 1100/4239 — Loss: 0.0657
  Batch 1200/4239 — Loss: 0.0681
  Batch 1300/4239 — Loss: 0.1326
  Batch 1400/4239 — Loss: 0.1110
  Batch 1500/4239 — Loss: 0.1051
  Batch 1600/4239 — Loss: 0.3976
  Batch 1700/4239 — Loss: 0.0391
  Batch 1800/4239 — Loss: 0.1650
  Batch 1900/4239 — Loss: 0.1111
  Batch 2000/4239 — Loss: 0.0820
  Batch 2100/4239 — Loss: 0.1097
  Batch 2200/4239 — Loss: 0.0791
  Batch 2300/4239 — Loss: 0.0862
  Batch 2400/4239 — Loss: 0.4986
  Batch 2500/4239 — Loss: 0.1536
  Batch 2600/4239 — Loss: 0.0649
  Batch 2700/4239 — Loss: 0.1700
  Batch 2800/4239 — Loss: 0.0603
  Ba

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ New best model saved! ROC-AUC: 0.9903


epoch,▁▅█
train_loss,█▃▁
val_f1,▇▁█
val_loss,█▁▂
val_roc_auc,▁▄█
epoch,3
train_loss,0.16001
val_f1,0.65184
val_loss,0.24677
val_roc_auc,0.99025



Training complete! Best ROC-AUC: 0.9903
